# EX 1- Jonas Gstoettenmayr

In [16]:
from bs4 import BeautifulSoup, UnicodeDammit
from urllib.robotparser import RobotFileParser
import requests
import json
from time import sleep
from copy import deepcopy
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

In [2]:
SOURCES: list[str] = ["https://arstechnica.com/", "https://techcrunch.com/", "https://www.techradar.com/"]
TERMS: list[str] = ["neural-net", "artifical-intelligence", "-ai-", "openai", "anthropic", "-ml-", "machine-learning", "-llm-", "keras", "pytorch", "mcp-server", "tensorflow", "claude", "gemini", "chatbot"]
USER_AGENT = "JOHN WEBSCRAPER"
VISITED: set[str] = set()
OUTPUT: str = "./output.json"

## Get Robots and Sitemaps

In [3]:
robots: list[RobotFileParser] = []
for source in SOURCES:
    r = RobotFileParser(source+"robots.txt")
    r.read()
    robots.append(r)
robots

In [4]:
sitemaps: list[str] = [site.site_maps() for site in robots] #type:ignore
# sitemaps[1] = [sitemaps[1][1]] #type:ignore reuters has many sitemaps i just need the one
sitemaps = [sites[0] for sites in sitemaps]#type:ignore
sitemaps

['https://arstechnica.com/sitemap.xml',
 'https://techcrunch.com/sitemap.xml',
 'https://www.techradar.com/sitemap.xml']

## Usefull Article Extraction

In [5]:
def map_to_robots(uri: str) -> int:
    for i, s in enumerate(SOURCES):
        if s in uri:
            return i
    return 0 # saftey i guess

In [6]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [ ]:
def get_subsitemaps(sitemapUrl: str) -> list[str]:
    if robots[map_to_robots(sitemapUrl)].can_fetch(USER_AGENT, sitemapUrl) == False:
        print("Not allowed to fetch!")
        return [""]
    resp = requests.get(sitemapUrl, headers=headers)
    soup = BeautifulSoup(resp.content, "xml") # Use "xml" for sitemaps
    return [link.text for link in soup.find_all("loc")]

submaps = [get_subsitemaps(sitemap) for sitemap in sitemaps]#type:ignore

In [8]:
GOOD_SITEMAPS = ["post-sitemap11", "post-sitemap12", "sitemap-202"]

In [9]:
filtered_links: list[set[str]] = [
    {link for link in links
    if any(word.lower() in link.lower() for word in GOOD_SITEMAPS)}
    for links in submaps 
]
filtered_links

[{'https://arstechnica.com/post-sitemap11.xml',
  'https://arstechnica.com/post-sitemap110.xml',
  'https://arstechnica.com/post-sitemap111.xml',
  'https://arstechnica.com/post-sitemap112.xml',
  'https://arstechnica.com/post-sitemap113.xml',
  'https://arstechnica.com/post-sitemap114.xml',
  'https://arstechnica.com/post-sitemap115.xml',
  'https://arstechnica.com/post-sitemap116.xml',
  'https://arstechnica.com/post-sitemap117.xml',
  'https://arstechnica.com/post-sitemap118.xml',
  'https://arstechnica.com/post-sitemap119.xml',
  'https://arstechnica.com/post-sitemap12.xml',
  'https://arstechnica.com/post-sitemap120.xml'},
 set(),
 {'https://www.techradar.com/sitemap-2020-01.xml',
  'https://www.techradar.com/sitemap-2020-02.xml',
  'https://www.techradar.com/sitemap-2020-03.xml',
  'https://www.techradar.com/sitemap-2020-04.xml',
  'https://www.techradar.com/sitemap-2020-05.xml',
  'https://www.techradar.com/sitemap-2020-06.xml',
  'https://www.techradar.com/sitemap-2020-07.xml',

In [10]:
articles: set[str] = set()
while len(articles) <180:
    print("searching ...", end="")
    if sum([len(x) for x in filtered_links]) == 0: break
    for site in filtered_links:
        if len(site) == 0: continue
        current_site = site.pop()
        subsite_article = get_subsitemaps(current_site)
        print(f"{current_site}", end=", ")
        for ar in subsite_article:
            for term in TERMS:
                if term in ar and ".jpg" not in ar:
                    articles.add(ar)
    print(f"\nsleepy ...(*￣０￣)ノ ... has {len(articles)}")
    sleep(1)
len(articles)

searching ...https://arstechnica.com/post-sitemap115.xml, https://www.techradar.com/sitemap-2022-12.xml, 
sleepy ...(*￣０￣)ノ ... has 111
searching ...https://arstechnica.com/post-sitemap118.xml, https://www.techradar.com/sitemap-2024-09.xml, 
sleepy ...(*￣０￣)ノ ... has 417


417

In [11]:
articles = {articles.pop() for _ in range(190)}

In [ ]:
TECHCRUNCH = "https://techcrunch.com/category/artificial-intelligence/page/"
counter = 2
while len(articles) < 200:
    uri = TECHCRUNCH+f"{counter}/"
    print(f"searching {uri}", end="")
    if robots[map_to_robots(uri)].can_fetch(USER_AGENT, uri) == False:
        print("Not allowed to fetch!")
        break
    requ =  requests.get(uri)
    soup = BeautifulSoup(requ.content, "lxml")
    target_divs = soup.find_all("div", class_="loop-card__content")
    for target in target_divs:
        link = target.find("a")
        if link:
            dest = link.get("href")
            for term in TERMS:
                if dest and term in dest and ".jpg" not in dest:
                    articles.add(dest)#type:ignore
    counter +=1
    print(f"\nsleepy ...(*￣０￣)ノ ... has {len(articles)}")
    sleep(1)
len(articles)

searching https://techcrunch.com/category/artificial-intelligence/page/2/
sleepy ...(*￣０￣)ノ ... has 195
searching https://techcrunch.com/category/artificial-intelligence/page/3/
sleepy ...(*￣０￣)ノ ... has 197
searching https://techcrunch.com/category/artificial-intelligence/page/4/
sleepy ...(*￣０￣)ノ ... has 204


204

In [13]:
backup = deepcopy(articles)

In [ ]:
article_split = [[a for a in articles if comp in a] for comp in SOURCES]

## Scraping the Articles and exproting to JSON

In [15]:
def from_uri_to_dict(uri: str):
    if robots[map_to_robots(uri)].can_fetch(USER_AGENT, uri) == False:
        print("Not allowed to fetch!")
    requ = requests.get(uri)
    soup = BeautifulSoup(requ.content, "lxml")
    dicc: dict[str, str] = {"timestamp": datetime.now().strftime("%d.%m.%Y %H:%M:%S")}
    dicc["URL"] = uri
    dicc["domain"] = SOURCES[map_to_robots(uri)]
    header = soup.find("body").find("h1") #type:ignore
    if header is not None: dicc["title"] = header.text
    else: dicc["title"] = "Unknown"
    pubDate = soup.find("time")
    if pubDate is not None: dicc["publication_date"] = pubDate.text
    else: dicc["publication_date"] = "Unknown"
    
    if dicc["domain"] == SOURCES[0]:
        dicc["text"] = "\n".join([line.text for line in soup.find_all("div", class_="post-content")])
    elif dicc["domain"] == SOURCES[1]:
        dicc["text"] = "\n".join([line.text for line in soup.find_all("div", class_="wp-block-post-content")])
    else:
        dicc["text"] = "\n".join([line.text for line in soup.find_all("div", class_="text-copy")])
    
    return dicc

In [21]:
def from_site_to_jsons(sites: list[str])->list[dict[str, str]]:
    scraped_sites = []
    for site in sites:
        try: 
            scraped_sites.append(from_uri_to_dict(site))
        except:
            print(f"failed at {site}")
            scraped_sites.append({"failed": f"{site}"})
        print(f"{site} is sleepy...")
        sleep(1)
    return scraped_sites

In [ ]:
with ThreadPoolExecutor(max_workers=len(SOURCES)) as executor:
    results = list(executor.map(from_site_to_jsons, article_split))

In [25]:
backup = deepcopy(results)

In [26]:
result = []
for res in results:
    result += res
len(result)

204

In [27]:
with open("./scraped_data.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4)

## Discussion

Short reflection discussing ethical considerations, challenges, and data quality issues

**Ethical considerations:**

 We have discuss those both in the lecture as well as in the ethics course last year, so I will keep it rather brief. For personal use I don't really see any problem with webscraping. For comercial use I think it should be up to the owner of the website to decide on the policy, which is why I am a big fan of the robots.txt. Of course one should also always respect the website one is scraping from as the answering of your queries slows the services for others and incures a cost on the provider. Furhtermore Webscrapers do not "view" adds, the primary income source of most free sites, which are the ones we can scrape, and as such they not only incur cost but do not earn anything for their servces, this should be kept in mind when scraping. To sum up, commercial scrapping should not only respect the robots.txt but should most likly be leagaly regulated so as to be beneficial for both the scraper and the one beeing scraped, e.g. google bringing people onto a website, for personal usage I don't think there are any problems.
 
**Challanges:**

The biggest challange I faced durring this EX was to find news sites that confirm to a few standards:

- Allowing me to access the articles throuh python
- having a reasonable sitemap

The next biggest challange came from finding a way to navigate the sitemaps in a way that worked for all of them.
The content specific filtering was rather easy with me simply searching for buzzwords inside of the title, which was thankfully made possible by the fact that all the news-sites I choose followed the same convention of putting the title in the link and splitting up words with a hyphon.

For extracting information from the articles they followed, mostly, the same convention for all parts but the content in which the classes which hold the content are nammed differentl and organised in wildly different manners.

The last challange was self inflicted with my want to run it in threads so that all 3 could be processed in parralled and only each site itself waits for 1 second not the combination of them.

**Quality issues:**

The biggest issue is that I scrape the entire content field which can include unrelated things.
This is especially visible for "techrader" where the start of the content is filled with the options of the newsletter and share options.
For better quality we definitly need cleaning of redunant and useless information as well as a closer looks into each webpage itself to better extract information from the tags.

The Pro websites of techrader are mixed into the sitemaps but cannot be accessed by the scraper (but are possible by the normal webbrowser?) and as such show up as a 403 error.
As this only effects 9 pages out of 204 I'll leave it in to showcase this phenomenon. This was also a small challange as the 403 page does not conform to news-article tags and as such required a more robust solution for extracting infor without crashing.